# SAKE CNN Master Notebook

This notebook is to: 
- Generate the instructions to run CNN training from your terminal
- Run testing (with threshold optimisation)
- Run Grad-CAM on outputs

## Setup
Always run the below cell first before any of the other cells!

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

#os.environ["CUDA_VISIBLE_DEVICES"] = "0"

seed    = 42 #38, 39, 40, 41, 42

wd.     = "./SAKE_scripts/deep_learning" # REPLACE with your full working directory path
datadir = os.path.join(wd, "data")
outdir  = os.path.join(wd, "outputs", f"seed{seed}")
if not os.path.exists(outdir):
    os.mkdir(outdir)

## Data Preparation

Split Train and Test sets according to the seed, stratified by gender & dx.

**NOTE**: This uses `SAKE_MASTER.csv`, which is a data summary csv containing, for each participant's T1-weighted MRI, their: 
- Unique identifier ('PTID')
- Sex ('PTGENDER', 0 = Female, 1 = Male)
- Age ('T1AGE')
- Diagnosis ('DIAGNOSIS', 0 = control, 1 = Alzheimer's Disease)
- The full path to their MRI nifti file ('T1_path', format example: '/path/to/images/mri.nii.gz')

In [ ]:
MASTERpath = os.path.join(datadir, "SAKE_MASTER.csv")
subinfo    = pd.read_csv(MASTERpath)

# Split dataframe to train and test set
subinfo['stratify_key'] = subinfo['PTGENDER'].astype(str) + '_' + subinfo['DIAGNOSIS'].astype(str)
train_cn, test_cn       = train_test_split(subinfo, test_size=0.15, stratify=subinfo['stratify_key'], random_state=seed) #stratify splits according to dx and sex
print("CN+AD for Training:", train_cn.shape[0])
print("CN+AD for Testing :", test_cn.shape[0])

# Save data splits as new csvs
train_cn.to_csv(os.path.join(outdir, "SAKE_train.csv"), index=False)
test_cn.to_csv(os.path.join(outdir, "SAKE_test.csv"), index=False)

## Model Training

Run model training:

- Open terminal
- `cd [WORKING DIRECTORY]`
- `conda activate [YOUR PYTORCH ENVIRONMENT]`
- Copy the command printed below and paste into your terminal's command line

In [ ]:
gpu        = 0 
model_name = "resnet50" #resnet18, resnet50, densenet121

print(
    f"CUDA_VISIBLE_DEVICES={gpu} $CONDA_PREFIX/bin/python SAKE_train.py "
    f"--dev gpu --seed {seed} --model_name {model_name} "
    f"--outdir {outdir} "
    f"--trainpath {os.path.join(outdir, 'SAKE_train.csv')} "
    f"--testpath {os.path.join(outdir, 'SAKE_test.csv')} "
    f"> {os.path.join(outdir, model_name + '_train.log')} 2>&1 &"
)

## Model Test results

Find optimal classification thresholds using Youden's Index, and apply to the test set. Aggregate the performances for each seed using all performance metrics.

In [ ]:
#With Threshold optimisation
import os
import torch
import pandas as pd
import numpy as np
import torchio as tio
import pytorch_lightning as pl
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import (roc_auc_score, roc_curve, accuracy_score,
                             precision_score, recall_score, f1_score,
                             confusion_matrix)
from tqdm import tqdm
from SAKE_train import NN3DClassifier

# --- Matching SAKE_train.py ---
image_size   = (109, 109, 109)
spacing      = (2, 2, 2)
batch_size   = 16
val_ratio    = 0.15
num_workers  = 8

seeds        = [38, 39, 40, 41, 42]
model_names  = ["resnet18", "resnet50", "densenet121"]
all_results  = []

def load_subjects(csv_path):
    """Mirrors BrainDataModule.load_subjects exactly."""
    img_data = pd.read_csv(csv_path)
    subjects = []
    for _, row in tqdm(img_data.iterrows(), total=len(img_data), desc='Loading'):
        subject = tio.Subject(
            image=tio.ScalarImage(row["T1_path"]),
            label=torch.tensor(int(row["DIAGNOSIS"]), dtype=torch.long),
            ptid=row["PTID"])
        subjects.append(subject)
    return subjects

def get_preprocessing_transform():
    """Mirrors BrainDataModule.preprocessing_transform + postprocessing — NO augmentation."""
    return tio.Compose([
        tio.ZNormalization(masking_method=lambda x: (x.data != 0).bool()),
        tio.Resample(spacing),
        tio.CropOrPad(image_size),
        tio.Mask(masking_method=lambda x: (x.data != 0).bool()),
    ])

for model_name in model_names:
    for seed in seeds:
        print(f"\n{'='*50}")
        print(f"  {model_name} | seed {seed}")
        print(f"{'='*50}")

        outdir     = os.path.join(wd, "outputs", f"seed{seed}")
        model_dir  = os.path.join(outdir, f"output_{model_name}")
        train_csv  = os.path.join(outdir, "SAKE_train.csv")

        # --- Reconstruct val split using EXACTLY the same method as training ---
        # pl.seed_everything was called before BrainDataModule in training,
        # but the split itself only depends on the torch.Generator seed (line 98).
        # We replicate just that generator call.
        subjects_train = load_subjects(train_csv)
        num_subs       = len(subjects_train)
        num_val_subs   = int(round(num_subs * val_ratio))
        num_train_subs = num_subs - num_val_subs

        _, val_subjects = random_split(
            subjects_train,
            [num_train_subs, num_val_subs],
            generator=torch.Generator().manual_seed(seed)  # matches line 98 exactly
        )

        transform  = get_preprocessing_transform()
        val_set    = tio.SubjectsDataset(val_subjects, transform=transform)
        val_loader = DataLoader(val_set, batch_size=batch_size,
                                num_workers=num_workers, shuffle=False)

        # --- Load checkpoint via Lightning (matches how testing was done) ---
        ckpt_path  = os.path.join(model_dir, "checkpoints", "best_model.ckpt")
        best_model = NN3DClassifier.load_from_checkpoint(
            ckpt_path,
            model_name=model_name,
            learning_rate=0.001,
            weight_decay=1e-4,
            num_classes=2
        )
        best_model.eval()
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        best_model = best_model.to(device)

        # --- Inference on val set ---
        all_probs, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                imgs    = batch['image'][tio.DATA].to(device)
                labels  = batch['label'].view(-1)
                logits  = best_model(imgs)
                probs   = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                all_probs.extend(probs)
                all_labels.extend(labels.numpy())

        all_probs  = np.array(all_probs)
        all_labels = np.array(all_labels)

        # --- Youden's Index on val probabilities ---
        fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
        youden_idx        = np.argmax(tpr - fpr)
        optimal_threshold = float(thresholds[youden_idx])
        val_auroc         = roc_auc_score(all_labels, all_probs)
        print(f"Val AUROC: {val_auroc:.4f} | Optimal threshold: {optimal_threshold:.4f}")

        # Save val predictions for future reference
        pd.DataFrame({
            "target"      : all_labels,
            "prob_class_1": all_probs,
            "pred"        : (all_probs >= optimal_threshold).astype(int),
        }).to_csv(os.path.join(model_dir, "predictions_val.csv"), index=False)

        # --- Apply threshold to saved test predictions ---
        test_preds  = pd.read_csv(os.path.join(model_dir, "predictions_test.csv"))
        y_test      = test_preds["target"].values
        y_test_prob = test_preds["prob_class_1"].values
        y_test_pred = (y_test_prob >= optimal_threshold).astype(int)
        
        # Update predictions csv
        test_preds["pred_thresholded"] = y_test_pred
        test_preds.to_csv(os.path.join(model_dir, "predictions_test.csv"), index=False)
        tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
        all_results.append({
            "model_name"        : model_name,
            "seed"              : seed,
            "optimal_threshold" : optimal_threshold,
            "val_auroc"         : val_auroc,
            "accuracy"          : accuracy_score(y_test, y_test_pred),
            "auc_roc"           : roc_auc_score(y_test, y_test_prob),
            "sensitivity"       : recall_score(y_test, y_test_pred, average="macro"),
            "precision"         : precision_score(y_test, y_test_pred, average="macro"),
            "specificity"       : tn / (tn + fp),
            "f1"                : f1_score(y_test, y_test_pred, average="macro"),
        })

# --- Collate and summarise ---
performance_df = pd.DataFrame(all_results)
summary = (performance_df
           .groupby("model_name")
           .agg(
               accuracy_mean    =("accuracy",          "mean"),
               accuracy_std     =("accuracy",          "std"),
               auc_roc_mean     =("auc_roc",           "mean"),
               auc_roc_std      =("auc_roc",           "std"),
               sensitivity_mean =("sensitivity",       "mean"),
               sensitivity_std  =("sensitivity",       "std"),
               specificity_mean =("specificity",       "mean"),
               specificity_std  =("specificity",       "std"),
               precision_mean   =("precision",         "mean"),
               precision_std    =("precision",         "std"),
               f1_mean          =("f1",                "mean"),
               f1_std           =("f1",                "std"),
               threshold_mean   =("optimal_threshold", "mean"),
               threshold_std    =("optimal_threshold", "std"),
           )
           .reset_index())

performance_df.to_csv(os.path.join(wd, "cnn_performances_thresholded.csv"), index=False)
summary.to_csv(os.path.join(wd, "cnn_performances_thresholded_summary.csv"), index=False)
print("\n=== SUMMARY (mean ± SD across seeds) ===")
print(summary.to_string(index=False))

## GradCAM

Generate Grad-CAM for each test image. Make average image for positive (AD) class.

> This uses the Grad-CAM functions from: https://github.com/jacobgil/pytorch-grad-cam

In [ ]:
####################
# GradCAM functions
####################

import matplotlib.pyplot as plt
from matplotlib import colormaps
import torch
import torchio as tio
import nibabel as nib
import torch.nn.functional as F
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def prep_gradcam_3d(img_path, trained_model, target_layer, outdir=None):
    """
    Prepare GradCAM overlays for 3D CNN.
    """
    # Load image, apply transforms and convert to tensor
    image_size = (109, 109, 109)
    spacing = (2, 2, 2)
    
    transform = tio.Compose([
        tio.ZNormalization(masking_method=lambda x: (x.data != 0).bool()),
        tio.Resample(spacing),
        tio.CropOrPad(image_size)])
    
    img = tio.Subject(image=tio.ScalarImage(img_path))
    img = transform(img)
    img_tensor = img.image.data.unsqueeze(0).cuda()
    
    # GradCAM setup
    trained_model.eval().cuda()
    cam = GradCAM(model=trained_model, target_layers=target_layer)
    outputs = trained_model(img_tensor)
    pred_class = outputs.argmax(dim=1).item()
    
    # Compute Grad-CAM map
    grayscale_cam = cam(input_tensor=img_tensor, targets=[ClassifierOutputTarget(pred_class)])
    grayscale_cam = grayscale_cam[0]  # shape (D, H, W)
    grayscale_cam = (grayscale_cam - grayscale_cam.min()) / (grayscale_cam.max() - grayscale_cam.min()) #Normalise
    
    #Save outputs
    if outdir is not None:
        cam_nii = nib.Nifti1Image(grayscale_cam, affine=np.eye(4))
        # add a bit to save as .nii.gz file
        # add a bit to save img_tensor
    
    return img_tensor, grayscale_cam

def visualise_gradcam_3d(img_tensor, grayscale_cam, slices=[30, 40, 50, 60, 70, 80], title="Grad-CAM", outdir=None):
    """
    Visualise GradCAM overlays for 3D CNN.
    """
    input_volume = img_tensor.squeeze().detach().cpu().numpy()  # shape (D, H, W)
    D, H, W = input_volume.shape

    heatmap_tensor = torch.from_numpy(grayscale_cam).float().unsqueeze(0).unsqueeze(0)
    upsampled_heatmap = F.interpolate(heatmap_tensor, size=(D, H, W), mode='trilinear', align_corners=False)
    heatmap_resized = upsampled_heatmap.squeeze().detach().cpu().numpy()  # shape (D, H, W)

    slices = slices
    alpha = 0.6
    cmap = colormaps['turbo']

    fig, axs = plt.subplots(1, len(slices), figsize=(18, 4))
    plt.subplots_adjust(wspace=0.01, hspace=0)

    heatmap_for_colorbar = None

    for ax, slice_idx in zip(axs, slices):
        image_slice = np.rot90(input_volume[slice_idx, :, :], k=1)
        heatmap_slice = np.rot90(heatmap_resized[slice_idx, :, :], k=1)

        # Normalise image
        image_norm = (image_slice - image_slice.min()) / (image_slice.max() - image_slice.min() + 1e-8)

        # Enhance and normalise heatmap
        heatmap_enhanced = heatmap_slice ** 2
        heatmap_rgb = cmap(heatmap_enhanced.clip(0, 1))[:, :, :3]

        # Overlay
        overlay = (1 - alpha) * np.stack([image_norm]*3, axis=-1) + alpha * heatmap_rgb
        ax.imshow(overlay)
        ax.set_title(f"Slice {slice_idx}", fontsize=10)
        ax.axis('off')

        # Save heatmap for colorbar
        if heatmap_for_colorbar is None:
            heatmap_for_colorbar = ax.imshow(heatmap_enhanced, cmap=cmap)
            heatmap_for_colorbar.set_visible(False)  # hide this slice

    # Add colorbar
    cbar = fig.colorbar(heatmap_for_colorbar, ax=axs, orientation='horizontal', fraction=0.05, pad=0.08)
    cbar.set_label('Activation Intensity')
    fig.suptitle(f'{title}', fontsize=16)
    
    if outdir is not None:
        fig.savefig(os.path.join(outdir, "gradcam_visualisation.png"), dpi=500, bbox_inches='tight')
    
    plt.show()

In [ ]:
# Identify the subs correctly classified as AD, and list their T1 paths
predictions   = pd.read_csv(os.path.join(outdir, f"output_{model_name}", "predictions_test.csv"))
test_sub_info = pd.read_csv(os.path.join(outdir, "SAKE_test.csv"))

tp       = predictions[(predictions["pred_thresholded"] == 1) & (predictions["target"] == 1)] # Keeping only true positives
tp_info  = tp.merge(test_sub_info[["PTID", "T1_path"]], left_on="ptid", right_on="PTID", how="inner")
tp_paths = tp_info["T1_path"].tolist()
print(f"Number of TP subjects: {len(tp_paths)}")

# Load trained ResNet50
from models.resnet3d import resnet50_3d

trained_model  = resnet50_3d(in_channels=1, num_classes=2)
checkpoint     = torch.load(os.path.join(outdir, f"output_{model_name}", "checkpoints", "best_model.ckpt"), map_location="cpu")
state_dict     = checkpoint["state_dict"]
new_state_dict = {}
for k, v in state_dict.items():
    name = k
    if name.startswith("model."):
        name = name.replace("model.", "")
    new_state_dict[name] = v
trained_model.load_state_dict(new_state_dict, strict=True)

target_layer = [trained_model.layer4[-1]] #For ResNet (18 & 50)

In [ ]:
# Generate mean Grad-CAM
sum_cam    = None
sum_sq_cam = None
n          = 0

for path in tp_paths:
    _, cam = prep_gradcam_3d(path, trained_model, target_layer)

    if sum_cam is None:
        sum_cam = np.zeros_like(cam)
        sum_sq_cam = np.zeros_like(cam)

    sum_cam += cam
    sum_sq_cam += cam ** 2
    n += 1

mean_cam = sum_cam / n
print(f"Averaged {n} Grad-CAM volumes")

In [ ]:
# Visualise on top of MNI standard
mni_path      = os.path.join("./SAKE_scripts/sake/ext/standard", "MNI152_T1_1mm_brain.nii.gz")
img_tensor, _ = prep_gradcam_3d(mni_path, trained_model, target_layer)
visualise_gradcam_3d(img_tensor, mean_cam, slices=[30, 40, 50, 60, 70, 80], title="Mean Grad-CAM for Positive Class", outdir=outdir)

# Save as nifti
# nib.save(nib.Nifti1Image(mean_cam, affine=np.eye(4)), os.path.join(outdir, f"output_{model_name}", "mean_gradcam_tp.nii.gz"))